<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day06_practice2_%EC%A3%BC%ED%83%9D%EA%B0%80%EA%B2%A9_%ED%9A%8C%EA%B7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 실제 데이터 - 주택 가격 예측 (회귀)

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 1. 데이터 - 미국 주택 1460채 x 81열 로드 + 결측치 확인 (캐글 "House Prices" 대회 데이터)
CSV_URL  = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/house_train.csv"
CSV_PATH = "house_train.csv"

def load_house():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH)

  try:
    df = pd.read_csv(CSV_URL)
    df.to_csv(CSV_PATH, index = False)
    print(f"다운로드 완료 → {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_house()

다운로드 완료 → house_train.csv 저장 (다음 실행부턴 재사용)


In [ ]:
print("모양:", df.shape)
print("타깃 SalePrice:", df["SalePrice"].min(), "-", df["SalePrice"].max(), "달러")

# 결측치 현황
null_cols = df.isnull().sum()
print(f"\n결측치가 있는 열: {(null_cols > 0).sum()}개")
print(null_cols[null_cols > 0].sort_values(ascending=False).head(5))

모양: (1460, 81)
타깃 SalePrice: 34900 - 755000 달러

결측치가 있는 열: 19개
PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
dtype: int64


In [ ]:
# 셀 2. 전처리 - 결측 채우기 + 범주형 원-핫 인코딩
# 숫자 열 -> 결측은 중앙값으로 채움
# 범주형 열 -> 원-핫 인코딩 (pd.get_dummies)

target = df["SalePrice"].values.astype(np.float32)
feats = df.drop(columns=["Id", "SalePrice"])

num_cols = feats.select_dtypes(include="number").columns # 숫자형 열 목록
cat_cols = feats.select_dtypes(exclude="number").columns # 범주형(문자) 열 목록
print(f"\n숫자 열 {len(num_cols)}개 / 범주형 열 {len(cat_cols)}개")

feats[num_cols] = feats[num_cols].fillna(feats[num_cols].median()) # 숫자 결측 -> 각 열의 중앙값으로 채움
feats = pd.get_dummies(feats, columns=list(cat_cols), dummy_na=False) # 범주형 -> 원-핫, dummy_na=False:NaN용 범주를 만들지 않음

X = feats.values.astype(np.float32) # 전처리가 끝난 df를 넘파일 배열로 변환
print("전처리 후 특징 수:", X.shape[1])
y = target / 10000.0 # 스케일 축소


숫자 열 36개 / 범주형 열 43개
전처리 후 특징 수: 287


In [ ]:
print(feats.head())

   MSSubClass  LotFrontage  LotArea  OverallQual  OverallCond  YearBuilt  \
0          60         65.0     8450            7            5       2003   
1          20         80.0     9600            6            8       1976   
2          60         68.0    11250            7            5       2001   
3          70         60.0     9550            7            5       1915   
4          60         84.0    14260            8            5       2000   

   YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  ...  SaleType_ConLw  \
0          2003       196.0         706           0  ...           False   
1          1976         0.0         978           0  ...           False   
2          2002       162.0         486           0  ...           False   
3          1970         0.0         216           0  ...           False   
4          2000       350.0         655           0  ...           False   

   SaleType_New  SaleType_Oth  SaleType_WD  SaleCondition_Abnorml  \
0         False  

In [ ]:
# 셀 3. 분할 + 표준화(70/15/15)
# 데이터를 학습 70% / 검증 15% / 테스트 15% 로 나눈다
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.3,
    random_state=42   #stratify 회귀는 없음
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5,
    random_state=42
)

scaler = StandardScaler()
def to_t(Xa, ya, fit=False):
  Xs = scaler.fit_transform(Xa) if fit else scaler.transform(Xa)
  return (torch.tensor(Xs, dtype=torch.float32).to(device),
          torch.tensor(ya, dtype=torch.float32).reshape(-1, 1).to(device))
X_tr_t, y_tr_t = to_t(X_tr, y_tr, fit=True)
X_val_t, y_val_t = to_t(X_val, y_val)
X_te_t, y_te_t = to_t(X_te, y_te)

print(f"학습 {len(X_tr)} / 검증 {len(X_val)} / 테스트 {len(X_te)}")

학습 1022 / 검증 219 / 테스트 219


In [ ]:
# 셀 4. 회귀 모델
in_dim = X.shape[1]
model = nn.Sequential(
    nn.Linear(in_dim, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1), #회귀라 마지막에 Sigmoid 없음, 값 그대로 예측
).to(device)

loss_fn = nn.MSELoss() # 회귀 손실: 평균제곱오차(MSE)
opt = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 셀 5. 학습


best_val, best_state, wait, patience = float("inf"), None, 0, 30 # 최고기록,최고모델,대기 참을성

for epoch in range(1000):
  model.train()
  loss = loss_fn(model(X_tr_t), y_tr_t)
  opt.zero_grad(); loss.backward(); opt.step()

  model.eval()
  with torch.no_grad():
    val_loss = loss_fn(model(X_val_t), y_val_t).item()

  if val_loss < best_val:
    best_val = val_loss
    best_state = {k: v.clone() for k, v in model.state_dict().items()} #가중치 '복사' 저장 - {"key": value}, 최고 모델의 가중치(weight)가 딕셔너리
    wait = 0
  else:   # best_val을 갱신을 못하면
    wait += 1
    if wait >= patience:
      print(f" EarlyStopping: {epoch+1} 애폭에서 중단 " f"(best 는 {epoch+1-patience} 애폭 근처)")
      break
  if (epoch + 1) % 50 == 0:
    print(f"epoch {epoch+1:3d} | train {loss.item():8.2f} | val {val_loss:8.2f}")

model.load_state_dict(best_state) #best 시점 가중치로 복원

epoch  50 | train    24.18 | val    25.78
epoch 100 | train     5.03 | val    12.43
epoch 150 | train     2.86 | val    10.50
epoch 200 | train     2.00 | val     9.78
epoch 250 | train     1.51 | val     9.45
epoch 300 | train     1.18 | val     9.23
epoch 350 | train     0.94 | val     9.07
epoch 400 | train     0.76 | val     8.94
epoch 450 | train     0.62 | val     8.83
epoch 500 | train     0.51 | val     8.71
epoch 550 | train     0.42 | val     8.62
epoch 600 | train     0.34 | val     8.53
epoch 650 | train     0.29 | val     8.48
 EarlyStopping: 696 애폭에서 중단 (best 는 666 애폭 근처)


<All keys matched successfully>

In [ ]:
# 셀 6. 평가
model.eval()
with torch.no_grad():
  pred = model(X_te_t).cpu().numpy().flatten() * 10000 #NumPy는 GPU Tensor를 바로 처리할 수 없기 때문, 배열을 1차원으로 쭉 펴준다
  true = y_te * 10000

mae = np.mean(np.abs(pred - true)) # MAE: 오차 절댓값의 평균
rmse = np.sqrt(np.mean((pred - true) ** 2)) #큰 오차에 더 민감

print(f"\nMAE (평균 절대 오차): {mae:,.0f} 달러")
print(f"RMSE (제곱근 평균 제곱 오차): {rmse:,.0f} 달러")
print(f"참고: 집값 중앙값은 {np.median(true):,.0f} 달러")

print("\n샘플 예측:")
for i in range(3):
  print(f" 실제 {true[i]:>9,.0f} -> 예측 {pred[i]:>9,.0f} (오차 {pred[i]-true[i]:+,.0f})")


MAE (평균 절대 오차): 19,745 달러
RMSE (제곱근 평균 제곱 오차): 30,595 달러
참고: 집값 중앙값은 150,000 달러

샘플 예측:
 실제   124,000 -> 예측   127,190 (오차 +3,190)
 실제   141,000 -> 예측   147,992 (오차 +6,992)
 실제   143,000 -> 예측   202,194 (오차 +59,194)
